# Podocyte damage vs depletion: data audit notebook

**Question:** does a low podocyte damage score in a kidney biopsy mean the podocytes are damaged, or that there are fewer of them?

Run order: top to bottom. Use "Restart" then "Run All" to check that everything still works.
Environment: `conda activate podo`. Last updated: 2026-11-05.

**Contents**
1. Setup
2. PDS gene list (42 mouse genes)
3. ERCB (GSE104948): download sample table
4. ERCB: audit (platform, groups, centres)
5. NEPTUNE (GSE197307, GSE219185): audit
6. Series headers of all datasets (papers, files, overlaps)
7. NEPTUNE counts (GSE254957): peek and match samples to groups
8. Ortholog mapping, mouse to human (NOT RUN YET)
9. Next steps

## 1. Setup
Finds the repo root (the folder that contains `environment.yml`) and works from there, so paths such as `data/...` always work.
Run this cell first in every session.

In [1]:
import os, io, gzip, time, urllib.request, urllib.parse
from pathlib import Path

import pandas as pd
import GEOparse

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "environment.yml").exists()), None)
assert ROOT is not None, "environment.yml not found: open this notebook from inside the repo"
os.chdir(ROOT)
print("Working folder:", Path.cwd())

for d in ["data/raw/geo_meta", "data/metadata", "data/reference", "notes"]:
    Path(d).mkdir(parents=True, exist_ok=True)

Working folder: /home/intissar/projects/podocyte-damage-vs-depletion


## 2. PDS gene list
- Source: Padvitski et al., Cell Genomics 2026 (supplementary table).
- File: `data/reference/pds_genes.csv`, columns `mouse_gene_symbol` and `direction`.
- Checks: 42 genes, no duplicates, only "up"/"down" (expected: 39 down, 3 up).
- Issue found earlier and fixed here: a stray space in a column name, so spaces are stripped.

In [2]:
genes = pd.read_csv("data/reference/pds_genes.csv")
genes.columns = genes.columns.str.strip()
genes["mouse_gene_symbol"] = genes["mouse_gene_symbol"].str.strip()
genes["direction"] = genes["direction"].str.strip().str.lower()

assert len(genes) == 42, f"expected 42 genes, got {len(genes)}"
assert genes["mouse_gene_symbol"].is_unique, "duplicate gene symbols"
assert set(genes["direction"]) == {"down", "up"}, "unexpected direction values"
print(genes["direction"].value_counts().to_string())

genes.to_csv("data/reference/pds_genes.csv", index=False)   # clean header written back
mouse = genes["mouse_gene_symbol"].tolist()
print("OK:", len(mouse), "genes, e.g.", mouse[:5])

direction
down    39
up       3
OK: 42 genes, e.g. ['Aifm3', 'Angptl2', 'Astn2', 'C1qtnf7', 'Cers6']


## 3. ERCB (GSE104948): download the sample table
GEOparse in `how="brief"` mode returns the per-sample table (platform, tissue, diagnosis) but **not** series-level information (title, papers, files), so those come from section 6.
The raw GEO download goes to `data/raw/` (ignored by Git); the sample table goes to `data/metadata/` (committed).

In [3]:
def fetch_sample_table(acc):
    gse = GEOparse.get_GEO(geo=acc, destdir="data/raw/geo_meta", how="brief", silent=True)
    meta = gse.phenotype_data
    meta.to_csv(f"data/metadata/{acc}_metadata.csv")
    return meta

meta = fetch_sample_table("GSE104948")
print("samples x columns:", meta.shape)
print(meta[["title", "platform_id", "characteristics_ch1.0.tissue",
            "characteristics_ch1.1.diagnosis"]].head().to_string())

samples x columns: (196, 32)
                   title platform_id  characteristics_ch1.0.tissue characteristics_ch1.1.diagnosis
GSM2810645   H1-Glom-DN1    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810646  H1-Glom-DN10    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810647   H1-Glom-DN2    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810648   H1-Glom-DN5    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810649   H1-Glom-DN8    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy


## 4. ERCB audit
Findings recorded here:
- 196 glomerular biopsies on two platforms: GPL22945 (71) and GPL24120 (125).
- **No eGFR, proteinuria, age or sex** in the public metadata.
- 21 samples have an **empty diagnosis**: these are the living donors, labelled `LD` only in the sample title.
- **Platform = centre:** site H7 is entirely GPL22945, sites H1-H5 entirely GPL24120. Compare groups **within one platform only**.
- Controls: GPL22945 has 18 living donors + 3 tumor nephrectomy; GPL24120 has only 3 + 2.
- Samples also exist in older series GSE47183 and GSE32591: do not combine.

In [5]:
meta = pd.read_csv("data/metadata/GSE104948_metadata.csv", index_col=0)

# group and centre are encoded in the sample title, e.g. "H7-Glom-LD_988"
t = meta["title"].str.strip()
meta["site"] = t.str.split("-").str[0]
meta["group"] = t.str.split("-").str[2].str.replace(r"[_\d]+$", "", regex=True)

lines = []
lines.append(f"samples: {len(meta)}")
lines.append("\nPLATFORM:\n" + meta["platform_id"].value_counts().to_string())
lines.append("\nTISSUE:\n" + meta["characteristics_ch1.0.tissue"].value_counts().to_string())
lines.append("\nDIAGNOSIS column (incl. missing):\n"
             + meta["characteristics_ch1.1.diagnosis"].value_counts(dropna=False).to_string())
lines.append("\nGROUP (from title) x PLATFORM:\n"
             + pd.crosstab(meta["group"], meta["platform_id"]).to_string())
lines.append("\nSITE x PLATFORM:\n"
             + pd.crosstab(meta["site"], meta["platform_id"]).to_string())

kw = ["gfr", "creat", "protein", "albumin", "age", "sex", "gender", "follow", "outcome", "esrd"]
hits = [c for c in meta.columns if any(k in c.lower() for k in kw)]
lines.append("\nPossible clinical columns: " + str(hits))

text = "\n".join(lines)
Path("notes/geo_audit_GSE104948.txt").write_text(text)
meta.to_csv("data/metadata/GSE104948_metadata_parsed.csv")
print(text)

samples: 196

PLATFORM:
platform_id
GPL24120    125
GPL22945     71

TISSUE:
characteristics_ch1.0.tissue
Glomeruli from kidney biopsy    196

DIAGNOSIS column (incl. missing):
characteristics_ch1.1.diagnosis
Systemic Lupus Erythematosus                                   32
IgA Nephropathy                                                27
ANCA Associated Vasculitis                                     22
NaN                                                            21
Membranous Glomerulonephropathy                                21
Focal Segmental Glomerular Sclerosis                           18
Hypertensive Nephropathy                                       15
Minimal Change Disease                                         14
Diabetic Nephropathy                                           12
Focal Segmental Glomerular Sclerosis/Minimal Change Disease     6
Tumor Nephrectomy                                               5
Thin Membrande Disease                                          3

## 5. NEPTUNE audit (GSE197307 and GSE219185)
- **GSE197307:** glomeruli, 282 samples (FSGS 93, MCD 89, membranous 62, other nephrotic syndrome 29, healthy donor 8, unspecified 1). Superseded by GSE254957.
- **GSE219185:** **tubulointerstitium**, not glomeruli (220 samples: FSGS 114, MCD 106). Not independent of other NEPTUNE sets. Negative control only.
- No clinical variables (eGFR, proteinuria) in either series.
- GSE254957 has **no sample table** in GEO (see section 7).

In [6]:
def audit(acc):
    meta = fetch_sample_table(acc)
    L = [f"=== {acc}: {meta.shape[0]} samples, {meta.shape[1]} columns"]
    L.append("platform: " + str(meta["platform_id"].value_counts().to_dict()))
    L.append("titles: " + "; ".join(meta["title"].head(5)))
    for c in [c for c in meta.columns if c.startswith("characteristics")]:
        vc = meta[c].value_counts(dropna=False)
        L.append(f"- {c} ({len(vc)} unique): "
                 + "; ".join(f"{k} [{v}]" for k, v in vc.head(8).items()))
    kw = ["gfr", "creat", "protein", "upcr", "albumin", "age", "sex", "gender",
          "race", "follow", "outcome", "esrd"]
    L.append("possible clinical columns: "
             + str([c for c in meta.columns if any(k in c.lower() for k in kw)]))
    return "\n".join(L)

text = "\n\n".join(audit(a) for a in ["GSE219185", "GSE197307"])
Path("notes/geo_audit_neptune.txt").write_text(text)
print(text)

=== GSE219185: 220 samples, 34 columns
platform: {'GPL18573': 220}
titles: Focal Segmental Glomerulosclerosis [Sample_1]; Focal Segmental Glomerulosclerosis [Sample_10]; Minimal Change Disease [Sample_100]; Focal Segmental Glomerulosclerosis [Sample_101]; Minimal Change Disease [Sample_102]
- characteristics_ch1.0.tissue (1 unique): Tubulointerstitium from human kidney biopsy [220]
- characteristics_ch1.1.disease (2 unique): Focal Segmental Glomerulosclerosis [114]; Minimal Change Disease [106]
possible clinical columns: []

=== GSE197307: 282 samples, 32 columns
platform: {'GPL18573': 282}
titles: Sample_1_LivingDonor; Sample_2_LivingDonor; Sample_3_LivingDonor; Sample_4_LivingDonor; Sample_5_LivingDonor
- characteristics_ch1.0.tissue (1 unique): Glomeruli from kidney biopsy [282]
- characteristics_ch1.1.disease (6 unique): Focal Segmental Glomerulosclerosis [93]; Minimal Change Disease [89]; Membranous Nephropathy [62]; Other Nephrotic Syndrome [29]; Healthy living transplant donor [

## 6. Series headers (papers, files, overlaps)
Read directly from GEO as text, with automatic retries (NCBI sometimes returns a temporary 502 error).
What this showed:
- **GSE254957 is a re-analysis of GSE197307** (same 282 samples): use GSE254957, never both.
- GSE219185 shares samples with other NEPTUNE sets.
- GSE104948 (ERCB) samples were also deposited as GSE47183 and GSE32591.
- Raw data and clinical data for NEPTUNE need an approved ancillary study request.
- Papers: ERCB 29724730, 40059827; NEPTUNE 35477557, 36442540, 38838223, 41659822 (titles still to be looked up in PubMed).

*Dead end removed from the old notebook:* reading these fields through `GEOparse` (`gse.metadata`) returned empty values in `brief` mode.

In [7]:
def get_header(acc, tries=4, wait=8):
    url = ("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
           f"?acc={acc}&targ=self&form=text&view=brief")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for i in range(1, tries + 1):
        try:
            return urllib.request.urlopen(req, timeout=60).read().decode("utf-8", "replace")
        except Exception as e:
            print(f"  {acc}: attempt {i}/{tries} failed ({e})")
            time.sleep(wait * i)
    return None

keep = ("!Series_title", "!Series_pubmed_id", "!Series_supplementary_file",
        "!Series_summary", "!Series_overall_design", "!Series_relation")

lines = []
for acc in ["GSE254957", "GSE197307", "GSE219185", "GSE104948"]:
    lines.append("=" * 60 + "\n" + acc)
    txt = get_header(acc)
    if txt is None:
        lines.append("FAILED after retries")
        continue
    for line in txt.splitlines():
        if line.startswith(keep):
            lines.append(line[:1200])

text = "\n".join(lines)
Path("notes/geo_series_headers.txt").write_text(text)
print(text)

GSE254957
!Series_title = Glomerular voom-transformed RNAseq counts from the Neptune Cohort and healthy living donors [re-analysis of GSE197307]
!Series_pubmed_id = 36442540
!Series_pubmed_id = 41659822
!Series_summary = Glomerular Transcriptome in the Nephrotic Syndrome study network cohort
!Series_overall_design = Please note that GSE197307 samples were re-analyzed as described below.
!Series_overall_design = Glomerular RNAseq profiles; Raw data (fastq) files are available upon approved ancillary study request from the NEPTUNE consortium (please see https://www.neptune-study.org/ancillary-studies; email contact: NEPTUNE-STUDY@umich.edu
!Series_overall_design = *data processing step:
!Series_overall_design = fastqScreen 0.11.4
!Series_overall_design = cutadapt 1.14 -m 50 -A CTGTCTCTTATA -a CTGTCTCTTATA
!Series_overall_design = STAR 2.5.2.a --bamRemoveDuplicatesType UniqueIdentical --twopassMode Basic --outFilterIntronMotifs RemoveNoncanonical --outSAMtype BAM SortedByCoordinate --outS

## 7. NEPTUNE counts (GSE254957): peek and match samples to groups
GSE254957 has no per-sample records, so:
1. read only the **header line** of the counts file (streamed, not the whole file);
2. match the sample names to the GSE197307 sample table to get each sample's disease group.

Findings: 283 header fields = 1 gene column + **282 samples**; genes are **Ensembl IDs**; values are **raw integer counts**; donor names contain the group, patient samples are bare `Sample_N` and need the GSE197307 table.
Result: 282 of 282 matched. Output: `data/metadata/GSE254957_samples_from_GSE197307.csv`.

In [8]:
url = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE254nnn/GSE254957/suppl/"
       "GSE254957_NEPTUNE_GSE197307_UpdatedCountData.txt.gz")
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=60) as resp:
    with gzip.GzipFile(fileobj=resp) as gz:
        header = gz.readline().decode("utf-8", "replace").rstrip("\n").split("\t")
        first = gz.readline().decode("utf-8", "replace").rstrip("\n").split("\t")

print("header fields:", len(header), "| first:", header[:3], "| last:", header[-3:])
print("first data row:", first[:6])

m197 = pd.read_csv("data/metadata/GSE197307_metadata.csv", index_col=0)
ids_matrix = [h.strip() for h in header[1:]]
ids_geo = m197["title"].str.strip().tolist()

only_matrix = sorted(set(ids_matrix) - set(ids_geo))
only_geo = sorted(set(ids_geo) - set(ids_matrix))
print("in matrix, not in GEO:", only_matrix[:10])
print("in GEO, not in matrix:", only_geo[:10])
assert not only_matrix and not only_geo, "sample names do not match"

samples = (m197.assign(sample_id=m197["title"].str.strip())
                [["sample_id", "characteristics_ch1.1.disease"]]
                .rename(columns={"characteristics_ch1.1.disease": "disease"}))
samples = samples[samples["sample_id"].isin(ids_matrix)]
samples.to_csv("data/metadata/GSE254957_samples_from_GSE197307.csv", index=False)
print("\nsaved", len(samples), "samples")
print(samples["disease"].value_counts().to_string())

header fields: 283 | first: ['ENSEMBL_gene_ID', 'Sample_1_LivingDonor', 'Sample_2_LivingDonor'] | last: ['Sample_319', 'Sample_337', 'Sample_339']
first data row: ['ENSG00000000003', '3767', '3707', '5463', '3023', '3639']
in matrix, not in GEO: []
in GEO, not in matrix: []

saved 282 samples
disease
Focal Segmental Glomerulosclerosis    93
Minimal Change Disease                89
Membranous Nephropathy                62
Other Nephrotic Syndrome              29
Healthy living transplant donor        8
Nephrotic Syndrome, unspecified        1


## 8. Ortholog mapping (mouse to human):
The score was built in mice; our data are human, so each of the 42 mouse genes needs a human version.
- Source: Ensembl BioMart. **Write the Ensembl release/date into `notes/log.md`.**
- Keep **one-to-one** orthologs; list genes with several or no matches.
- NEPTUNE needs the **Ensembl ID**, ERCB needs the **Entrez ID**; keep the human symbol too.
- Report how many of the 42 are usable (goes into the README and the pre-registration).
- Output: `data/reference/pds_orthologs_raw.csv`, then a cleaned table.

In [9]:
def biomart(xml, tries=3):
    url = "https://www.ensembl.org/biomart/martservice?query=" + urllib.parse.quote(xml)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for i in range(tries):
        try:
            txt = urllib.request.urlopen(req, timeout=120).read().decode()
            return pd.read_csv(io.StringIO(txt), sep="\t")
        except Exception as e:
            print("retry", i + 1, e)
            time.sleep(8 * (i + 1))
    raise RuntimeError("BioMart failed")

query = """<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query virtualSchemaName="default" formatter="TSV" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="mmusculus_gene_ensembl" interface="default">
<Filter name="external_gene_name" value="%s"/>
<Attribute name="external_gene_name"/>
<Attribute name="ensembl_gene_id"/>
<Attribute name="hsapiens_homolog_ensembl_gene"/>
<Attribute name="hsapiens_homolog_associated_gene_name"/>
<Attribute name="hsapiens_homolog_orthology_type"/>
<Attribute name="hsapiens_homolog_orthology_confidence"/>
</Dataset></Query>""" % ",".join(mouse)

orth = biomart(query)
orth.columns = ["mouse_symbol", "mouse_ensembl", "human_ensembl", "human_symbol",
                "orthology_type", "confidence"]
orth.to_csv("data/reference/pds_orthologs_raw.csv", index=False)

print(orth.shape)
print(orth["orthology_type"].value_counts(dropna=False).to_string())
print("\nNot returned at all:", sorted(set(mouse) - set(orth["mouse_symbol"])))

(42, 6)
orthology_type
ortholog_one2one    42

Not returned at all: []


## 8b. Check the ortholog table
All 42 mouse genes returned a one-to-one human match. Check by eye: do the names look sensible (for example Wt1 to WT1, Zfp423 to ZNF423), is anything missing or shared, and is the confidence high?

In [10]:
orth = pd.read_csv("data/reference/pds_orthologs_raw.csv")
tab = genes.merge(orth, left_on="mouse_gene_symbol", right_on="mouse_symbol", how="left")

print("rows:", len(tab))
print("missing human Ensembl ID:", tab["human_ensembl"].isna().sum())
print("human genes hit by more than one mouse gene:", tab["human_ensembl"].duplicated(keep=False).sum())
print("confidence values:", tab["confidence"].value_counts(dropna=False).to_dict())
print(tab[["mouse_gene_symbol", "direction", "human_symbol",
           "human_ensembl", "confidence"]].to_string())

rows: 42
missing human Ensembl ID: 0
human genes hit by more than one mouse gene: 0
confidence values: {1: 42}
   mouse_gene_symbol direction human_symbol    human_ensembl  confidence
0              Aifm3      down        AIFM3  ENSG00000183773           1
1            Angptl2      down      ANGPTL2  ENSG00000136859           1
2              Astn2      down        ASTN2  ENSG00000148219           1
3            C1qtnf7      down      C1QTNF7  ENSG00000163145           1
4              Cers6      down        CERS6  ENSG00000172292           1
5              Cldn5      down        CLDN5  ENSG00000184113           1
6              Clic3      down        CLIC3  ENSG00000169583           1
7              Dach1      down        DACH1  ENSG00000276644           1
8                Ddn      down          DDN  ENSG00000181418           1
9             Dusp15      down       DUSP15  ENSG00000149599           1
10             Enpep      down        ENPEP  ENSG00000138792           1
11           

## 8c. Add Entrez IDs
NEPTUNE uses Ensembl IDs; ERCB uses Entrez IDs. Ask BioMart for the Entrez ID of each human gene.

In [11]:
ens_list = ",".join(tab["human_ensembl"].dropna().unique())
query2 = """<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query virtualSchemaName="default" formatter="TSV" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="hsapiens_gene_ensembl" interface="default">
<Filter name="ensembl_gene_id" value="%s"/>
<Attribute name="ensembl_gene_id"/>
<Attribute name="entrezgene_id"/>
</Dataset></Query>""" % ens_list

ent = biomart(query2)
ent.columns = ["human_ensembl", "human_entrez"]
print("rows returned:", len(ent), "| Ensembl IDs with several Entrez IDs:",
      ent["human_ensembl"].duplicated().sum())
print("without Entrez ID:", ent["human_entrez"].isna().sum())

ent = ent.dropna().drop_duplicates("human_ensembl")   # keep the first Entrez ID per gene
ent["human_entrez"] = ent["human_entrez"].astype("Int64")
tab = tab.drop(columns=["human_entrez"], errors="ignore").merge(ent, on="human_ensembl", how="left")
print(tab[["human_symbol", "human_ensembl", "human_entrez"]].head(10).to_string())

rows returned: 42 | Ensembl IDs with several Entrez IDs: 0
without Entrez ID: 0
  human_symbol    human_ensembl  human_entrez
0        AIFM3  ENSG00000183773        150209
1      ANGPTL2  ENSG00000136859         23452
2        ASTN2  ENSG00000148219         23245
3      C1QTNF7  ENSG00000163145        114905
4        CERS6  ENSG00000172292        253782
5        CLDN5  ENSG00000184113          7122
6        CLIC3  ENSG00000169583          9022
7        DACH1  ENSG00000276644          1602
8          DDN  ENSG00000181418         23109
9       DUSP15  ENSG00000149599        128853


## 8d. Gene coverage per dataset
Downloads the NEPTUNE counts file (about tens of MB) and the two ERCB matrices into `data/raw/` (ignored by Git), reads only the gene ID column, and checks which of our genes are measured. Saves the final working table `data/reference/pds_genes_human.csv`.

In [12]:
import shutil

def download(url, dest, tries=3):
    if Path(dest).exists():
        return
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for i in range(1, tries + 1):
        try:
            with urllib.request.urlopen(req, timeout=300) as r, open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
            return
        except Exception as e:
            print("retry", i, e)
            time.sleep(8 * i)
    raise RuntimeError("download failed: " + url)

# NEPTUNE (Ensembl IDs)
nep_path = "data/raw/GSE254957_counts.txt.gz"
download("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE254nnn/GSE254957/suppl/"
         "GSE254957_NEPTUNE_GSE197307_UpdatedCountData.txt.gz", nep_path)
nep_ids = (pd.read_csv(nep_path, sep="\t", usecols=[0]).iloc[:, 0]
             .astype(str).str.replace(r"\.\d+$", "", regex=True))
print("NEPTUNE genes measured:", len(nep_ids), "| example:", nep_ids.head(3).tolist())
tab["in_neptune"] = tab["human_ensembl"].isin(set(nep_ids))

# ERCB (Entrez IDs, one matrix per platform)
tab["human_entrez_str"] = tab["human_entrez"].astype("string")
for gpl in ["GPL22945", "GPL24120"]:
    p = f"data/raw/GSE104948-{gpl}_series_matrix.txt.gz"
    download("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE104nnn/GSE104948/matrix/"
             f"GSE104948-{gpl}_series_matrix.txt.gz", p)
    ids = (pd.read_csv(p, sep="\t", comment="!", usecols=[0]).iloc[:, 0]
             .astype(str).str.replace(r"_at$", "", regex=True))
    print(gpl, "probe sets:", len(ids), "| example IDs:", ids.head(3).tolist())
    tab[f"in_ercb_{gpl}"] = tab["human_entrez_str"].isin(set(ids))

cols = ["mouse_gene_symbol", "direction", "human_symbol", "human_ensembl", "human_entrez",
        "in_neptune", "in_ercb_GPL22945", "in_ercb_GPL24120"]
final = tab[cols]
final.to_csv("data/reference/pds_genes_human.csv", index=False)

print("\nUsable genes (of 42):")
print(final[["in_neptune", "in_ercb_GPL22945", "in_ercb_GPL24120"]].sum().to_string())
print("\nNot found in at least one dataset:")
print(final[~final[["in_neptune", "in_ercb_GPL22945", "in_ercb_GPL24120"]].all(axis=1)].to_string())

NEPTUNE genes measured: 51534 | example: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419']
GPL22945 probe sets: 12074 | example IDs: ['10000', '10001', '10002']
GPL24120 probe sets: 12074 | example IDs: ['10000', '10001', '10002']

Usable genes (of 42):
in_neptune          42
in_ercb_GPL22945    34
in_ercb_GPL24120    34

Not found in at least one dataset:
   mouse_gene_symbol direction human_symbol    human_ensembl  human_entrez  in_neptune  in_ercb_GPL22945  in_ercb_GPL24120
0              Aifm3      down        AIFM3  ENSG00000183773        150209        True             False             False
3            C1qtnf7      down      C1QTNF7  ENSG00000163145        114905        True             False             False
9             Dusp15      down       DUSP15  ENSG00000149599        128853        True             False             False
18            Metrnl      down       METRNL  ENSG00000176845        284207        True             False             False
24            Pard